In [1]:
# =============================================================================
# CELL 1: Imports and Configuration
# =============================================================================
# Run this cell first. If any import fails, install the missing package:
#   pip install sentence-transformers chromadb faster-whisper pyttsx3 PyPDF2 requests

# --- Standard library ---
import os
import re
import json
import uuid
import tempfile
from pathlib import Path

# --- HTTP / API communication ---
import requests

# --- PDF text extraction ---
from PyPDF2 import PdfReader

# --- Embeddings ---
from sentence_transformers import SentenceTransformer

# --- Vector store ---
import chromadb
from chromadb.config import Settings

# --- Speech-to-Text (loaded later in Cell 10+) ---
from faster_whisper import WhisperModel

# --- Text-to-Speech (loaded later in Cell 10+) ---
import pyttsx3

# --- Notebook utilities ---
from IPython.display import Audio, display

# =============================================================================
# CONFIGURATION — All constants in one place for easy tuning
# =============================================================================

# -- Ollama --
OLLAMA_BASE_URL = "http://localhost:11434"
OLLAMA_MODEL = "mistral"  # Change to "llama3" etc. if you've pulled another model

# -- Embedding model --
# BAAI/bge-small-en-v1.5: 384 dimensions, ~127MB, better retrieval than MiniLM
# Uses sentence-transformers — drop-in replacement, same API
# For retrieval: prefix queries with "Represent this sentence: " (handled in retrieval cell)
EMBEDDING_MODEL_NAME = "BAAI/bge-small-en-v1.5"
EMBEDDING_DIMENSIONS = 384  # Explicit so other cells can reference this

# -- Chunking --
CHUNK_SIZE = 800          # Max characters per chunk (up from 500 — more context per chunk)
CHUNK_OVERLAP = 100       # Overlap between consecutive chunks (scaled with chunk size)
CHUNK_MIN_WORDS = 20      # Quality filter: minimum word count to keep a chunk
CHUNK_MIN_LETTER_RATIO = 0.40  # Quality filter: minimum ratio of letters to total characters

# -- Retrieval --
TOP_K = 5                 # Number of chunks to retrieve per query
CHROMA_COLLECTION = "rag_documents"
CHROMA_PERSIST_DIR = "./chroma_data"

# -- Document classification --
# Keywords that signal the start of real content in academic documents
ACADEMIC_CONTENT_MARKERS = [
    "abstract", "introduction", "background", "chapter 1",
    "1. introduction", "1 introduction", "literature review"
]
# Keywords in early pages that suggest an academic document
ACADEMIC_SIGNALS = [
    "thesis", "dissertation", "submitted in partial fulfillment",
    "degree of", "bachelor", "master", "doctor of philosophy",
    "university", "department of", "supervisor", "declaration",
    "certificate", "acknowledgement", "acknowledgment"
]
ACADEMIC_SIGNAL_THRESHOLD = 2  # Minimum matches to classify as academic

# -- Whisper STT --
WHISPER_MODEL_SIZE = "base"

# -- Conversation history --
MAX_HISTORY_TURNS = 5

# -- File paths --
AUDIO_CACHE_DIR = "./audio_cache"
UPLOAD_DIR = "./uploads"
os.makedirs(AUDIO_CACHE_DIR, exist_ok=True)
os.makedirs(UPLOAD_DIR, exist_ok=True)
os.makedirs(CHROMA_PERSIST_DIR, exist_ok=True)

# =============================================================================
# VERIFICATION
# =============================================================================
print("✓ All imports successful")
print(f"  Ollama URL:        {OLLAMA_BASE_URL}")
print(f"  LLM Model:         {OLLAMA_MODEL}")
print(f"  Embedding Model:   {EMBEDDING_MODEL_NAME} ({EMBEDDING_DIMENSIONS}d)")
print(f"  Chunk size:        {CHUNK_SIZE} chars, {CHUNK_OVERLAP} overlap")
print(f"  Quality filter:    min {CHUNK_MIN_WORDS} words, min {CHUNK_MIN_LETTER_RATIO:.0%} letters")
print(f"  Top-K retrieval:   {TOP_K}")
print(f"  Whisper model:     {WHISPER_MODEL_SIZE}")
print(f"  ChromaDB dir:      {CHROMA_PERSIST_DIR}")

# Quick Ollama connectivity check
try:
    response = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=5)
    models = [m["name"] for m in response.json().get("models", [])]
    print(f"  Ollama models:     {', '.join(models) if models else 'No models found'}")
except requests.ConnectionError:
    print("  ⚠ Ollama not reachable — make sure it's running")

c:\Users\HP\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ All imports successful
  Ollama URL:        http://localhost:11434
  LLM Model:         mistral
  Embedding Model:   BAAI/bge-small-en-v1.5 (384d)
  Chunk size:        800 chars, 100 overlap
  Quality filter:    min 20 words, min 40% letters
  Top-K retrieval:   5
  Whisper model:     base
  ChromaDB dir:      ./chroma_data
  Ollama models:     mistral:latest


In [2]:
# =============================================================================
# CELL 2: Ollama Connection Test
# =============================================================================
# Verifies:
#   1. /api/generate works (single prompt → single completion)
#   2. /api/chat works (conversation-style with messages array)
#   3. The RAG prompt format we'll use in Cell 9 produces grounded answers
#
# If this cell fails, make sure Ollama is running: `ollama serve`
# and that your model is pulled: `ollama pull mistral`

# --- Test 1: /api/generate (single prompt) ---
print("=" * 60)
print("TEST 1: /api/generate")
print("=" * 60)

response = requests.post(
    f"{OLLAMA_BASE_URL}/api/generate",
    json={
        "model": OLLAMA_MODEL,
        "prompt": "What is retrieval augmented generation? Answer in two sentences.",
        "stream": False
    }
)

data = response.json()
print(f"Response: {data['response']}")
print(f"Tokens:   {data.get('eval_count', 'N/A')}")
print(f"Time:     {data.get('total_duration', 0) / 1e9:.2f}s")

# --- Test 2: /api/chat (conversation style — this is what our RAG pipeline uses) ---
print(f"\n{'=' * 60}")
print("TEST 2: /api/chat")
print("=" * 60)

response = requests.post(
    f"{OLLAMA_BASE_URL}/api/chat",
    json={
        "model": OLLAMA_MODEL,
        "messages": [
            {"role": "system", "content": "You are a helpful assistant. Keep answers brief."},
            {"role": "user", "content": "Explain what an embedding is in one sentence."}
        ],
        "stream": False
    }
)

data = response.json()
print(f"Response: {data['message']['content']}")
print(f"Tokens:   {data.get('eval_count', 'N/A')}")
print(f"Time:     {data.get('total_duration', 0) / 1e9:.2f}s")

# --- Test 3: Simulated RAG prompt format ---
# This mirrors exactly how Cell 9's rag_query() will structure the prompt:
# system message enforces grounding, user message contains context + question.
print(f"\n{'=' * 60}")
print("TEST 3: Simulated RAG prompt")
print("=" * 60)

fake_context = """
Company Policy Document:
Employees are entitled to 21 days of paid leave per calendar year.
Unused leave cannot be carried over to the next year.
Leave requests must be submitted at least two weeks in advance.
"""

user_question = "How many vacation days do I get?"

response = requests.post(
    f"{OLLAMA_BASE_URL}/api/chat",
    json={
        "model": OLLAMA_MODEL,
        "messages": [
            {
                "role": "system",
                "content": (
                    "You are a helpful assistant. Answer the user's question based ONLY on the "
                    "provided context. If the context doesn't contain the answer, say so. "
                    "Do not make up information."
                )
            },
            {
                "role": "user",
                "content": f"Context:\n{fake_context}\n\nQuestion: {user_question}"
            }
        ],
        "stream": False
    }
)

data = response.json()
print(f"Question: {user_question}")
print(f"Response: {data['message']['content']}")

print("\n✓ Ollama connection verified — all three tests passed")

TEST 1: /api/generate
Response:  Retrieval Augmented Generation (RAG) is a method in artificial intelligence that combines text generation with information retrieval. The goal is to enhance the model's responses by allowing it to retrieve relevant information from a pre-defined dataset during the conversation, rather than solely relying on its internal knowledge. This approach aims to improve the model's accuracy, completeness, and contextual understanding in the generated responses.
Tokens:   88
Time:     5.76s

TEST 2: /api/chat
Response:  An embedding is a representation that transforms data from its original space to a new space where it becomes more amenable to machine learning algorithms.
Tokens:   29
Time:     0.67s

TEST 3: Simulated RAG prompt
Question: How many vacation days do I get?
Response:  Based on the provided context, you are entitled to 21 days of paid leave per calendar year.

✓ Ollama connection verified — all three tests passed


In [3]:
# =============================================================================
# CELL 3: Document Loading and Text Extraction
# =============================================================================
# First step of the ingestion pipeline:
#   Raw PDF file → extracted plain text + document ID
#
# Changes from original:
#   - Returns a dict with doc_id, filename, and text (not just text)
#   - doc_id is used downstream to namespace chunks in ChromaDB,
#     enabling per-document deletion and multi-document support
#   - Removed TXT support for now (we're focusing on PDFs only)

def extract_text_from_pdf(file_path: str) -> str:
    """
    Extract all text from a PDF file, page by page.
    Returns the full extracted text as a single string.
    """
    reader = PdfReader(file_path)
    pages_text = []

    for i, page in enumerate(reader.pages):
        text = page.extract_text()
        if text:
            pages_text.append(text)
        else:
            print(f"  ⚠ Page {i+1}: No text extracted (might be an image or blank)")

    return "\n\n".join(pages_text)


def load_document(file_path: str) -> dict:
    """
    Load a PDF document and return its text with metadata.

    Args:
        file_path: Path to the PDF file

    Returns:
        dict with:
            - doc_id:   Unique identifier for this document
            - filename: Original filename
            - text:     Extracted text content
    """
    path = Path(file_path)

    if not path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")

    if path.suffix.lower() != ".pdf":
        raise ValueError(f"Unsupported format: {path.suffix}. Only .pdf is supported.")

    print(f"Loading PDF: {path.name}")
    text = extract_text_from_pdf(file_path)

    # Generate a unique doc_id — used to namespace chunks in ChromaDB
    # so we can later delete/re-ingest individual documents
    doc_id = uuid.uuid4().hex[:12]

    return {
        "doc_id": doc_id,
        "filename": path.name,
        "text": text
    }


# =============================================================================
# TEST: Load your sample document
# =============================================================================
SAMPLE_DOC_PATH = "./sample_doc.pdf"

doc = load_document(SAMPLE_DOC_PATH)

print(f"\n✓ Text extracted successfully")
print(f"  Doc ID:           {doc['doc_id']}")
print(f"  Filename:         {doc['filename']}")
print(f"  Total characters: {len(doc['text']):,}")
print(f"  Approx tokens:    ~{len(doc['text']) // 4:,}")
print(f"  Total pages:      {len(PdfReader(SAMPLE_DOC_PATH).pages)}")

print(f"\n{'=' * 60}")
print("PREVIEW (first 1000 characters):")
print("=" * 60)
print(doc["text"][:1000])
print("..." if len(doc["text"]) > 1000 else "")

Loading PDF: sample_doc.pdf

✓ Text extracted successfully
  Doc ID:           837d40253990
  Filename:         sample_doc.pdf
  Total characters: 60,719
  Approx tokens:    ~15,179
  Total pages:      46

PREVIEW (first 1000 characters):
 
1 
 Distributed Emotion Detection System  
Major Project submitted to  
Amity School of Engineering and Technology, Amity 
University  
In partial fulfillment for the requirements for the degree  of 
 
I. [B. Tech + M. Tech] in Artificial Intelligence and Robotics  
From  
Department of Artificial Intelligence, Amity School of 
Engineering and Technology  
 
By 
 
Angshuman Basu (A910112520006)  
 
Under the Guidance of  
Prof. Pronaya Bhattacharya  (309881 ) 
 
 
Amity School of Engineering and Technology  
Amity University  
Kolkata – 700135  
West Bengal, India  
    3rd December  2024  
  


 
2 
 Amity School of Engineering and Technology  
Amity University  
 
 
 
Certificate  
 
This is to certify that the project work entitled “ Distributed 

In [4]:
# =============================================================================
# CELL 4: Text Chunking with Quality Filter
# =============================================================================
# Splits extracted text into overlapping chunks, then filters out noise.
#
# Changes from original:
#   - Chunk size increased to 800 (from 500) for more context per chunk
#   - Overlap increased to 100 (from 50) to match
#   - Added secondary split for oversized sentences (clause boundaries)
#   - Quality filter (from old Cell 7b) is now baked in — no separate step
#   - Returns only clean, high-quality chunks ready for embedding

def split_oversized_sentence(sentence: str, chunk_size: int) -> list[str]:
    """
    Split a sentence that exceeds chunk_size on clause boundaries.
    Falls back to mid-point split if no clause boundary is found.

    Clause boundaries: semicolons, commas followed by space, " - ", " — "
    """
    if len(sentence) <= chunk_size:
        return [sentence]

    # Try splitting on clause boundaries
    split_points = []
    for pattern in ["; ", ", ", " - ", " — "]:
        for match in re.finditer(re.escape(pattern), sentence):
            split_points.append(match.start() + len(pattern))

    if split_points:
        # Pick the split point closest to the middle
        mid = len(sentence) // 2
        best = min(split_points, key=lambda x: abs(x - mid))
        left = sentence[:best].strip()
        right = sentence[best:].strip()
    else:
        # No clause boundary found — split at midpoint on a space
        mid = len(sentence) // 2
        space_pos = sentence.rfind(" ", 0, mid + 50)
        if space_pos == -1:
            space_pos = mid
        left = sentence[:space_pos].strip()
        right = sentence[space_pos:].strip()

    # Recurse in case either half is still oversized
    return split_oversized_sentence(left, chunk_size) + \
           split_oversized_sentence(right, chunk_size)


def is_quality_chunk(chunk: str) -> bool:
    """
    Filter out low-content chunks (title pages, signature lines, etc.)
    Uses CHUNK_MIN_WORDS and CHUNK_MIN_LETTER_RATIO from Cell 1 config.
    """
    words = [w for w in chunk.split() if len(w) > 1 and any(c.isalpha() for c in w)]
    if len(words) < CHUNK_MIN_WORDS:
        return False

    letters = sum(1 for c in chunk if c.isalpha())
    if len(chunk) == 0:
        return False
    if letters / len(chunk) < CHUNK_MIN_LETTER_RATIO:
        return False

    return True


def chunk_text(
    text: str,
    chunk_size: int = CHUNK_SIZE,
    overlap: int = CHUNK_OVERLAP,
    apply_quality_filter: bool = True
) -> list[str]:
    """
    Split text into overlapping chunks, respecting sentence boundaries.

    Strategy:
        1. Split text into sentences
        2. Break oversized sentences on clause boundaries
        3. Accumulate sentences into chunks up to chunk_size
        4. Overlap by rewinding a few sentences at chunk boundaries
        5. Filter out low-quality chunks (optional)

    Args:
        text:                 The full document text
        chunk_size:           Maximum characters per chunk
        overlap:              Approximate overlap in characters
        apply_quality_filter: If True, remove low-content chunks

    Returns:
        List of text chunks
    """
    # -- Step 1: Split into sentences --
    sentences = re.split(r'(?<=[.!?])\s+|\n\n+', text)
    sentences = [s.strip() for s in sentences if s.strip()]

    # -- Step 2: Handle oversized sentences --
    processed = []
    for s in sentences:
        processed.extend(split_oversized_sentence(s, chunk_size))
    sentences = processed

    # -- Step 3: Accumulate sentences into chunks --
    chunks = []
    current_chunk = []
    current_length = 0

    for sentence in sentences:
        sentence_length = len(sentence)

        # Would adding this sentence exceed chunk size?
        if current_length + sentence_length + 1 > chunk_size and current_chunk:
            chunks.append(" ".join(current_chunk))

            # Create overlap by rewinding
            overlap_sentences = []
            overlap_length = 0
            for s in reversed(current_chunk):
                if overlap_length + len(s) > overlap:
                    break
                overlap_sentences.insert(0, s)
                overlap_length += len(s) + 1

            current_chunk = overlap_sentences
            current_length = overlap_length

        current_chunk.append(sentence)
        current_length += sentence_length + 1

    # Don't forget the last chunk
    if current_chunk:
        chunks.append(" ".join(current_chunk))

    # -- Step 4: Quality filter --
    if apply_quality_filter:
        before = len(chunks)
        chunks = [c for c in chunks if is_quality_chunk(c)]
        removed = before - len(chunks)
        if removed > 0:
            print(f"  Quality filter removed {removed} low-content chunk(s)")

    return chunks


# =============================================================================
# TEST: Chunk the extracted text
# =============================================================================
chunks = chunk_text(doc["text"])

print(f"\n✓ Chunking complete")
print(f"  Total chunks:     {len(chunks)}")
print(f"  Chunk size range: {min(len(c) for c in chunks)} - {max(len(c) for c in chunks)} chars")
print(f"  Average size:     {sum(len(c) for c in chunks) // len(chunks)} chars")

# -- Show first 3 chunks --
for i, chunk in enumerate(chunks[:3]):
    print(f"\n{'=' * 60}")
    print(f"CHUNK {i+1} ({len(chunk)} chars):")
    print("=" * 60)
    print(chunk)

# -- Verify no oversized chunks --
oversized = [c for c in chunks if len(c) > CHUNK_SIZE + 100]  # Small tolerance for overlap
if oversized:
    print(f"\n⚠ {len(oversized)} chunks exceed {CHUNK_SIZE}+100 chars (max: {max(len(c) for c in oversized)})")
else:
    print(f"\n✓ No oversized chunks — all within target range")

# -- Verify overlap --
if len(chunks) >= 2:
    print(f"\n{'=' * 60}")
    print("OVERLAP CHECK:")
    print("=" * 60)
    print(f"End of Chunk 1:    ...{chunks[0][-100:]}")
    print(f"Start of Chunk 2:  {chunks[1][:100]}...")

  Quality filter removed 6 low-content chunk(s)

✓ Chunking complete
  Total chunks:     82
  Chunk size range: 301 - 799 chars
  Average size:     686 chars

CHUNK 1 (595 chars):
1 
 Distributed Emotion Detection System  
Major Project submitted to  
Amity School of Engineering and Technology, Amity 
University  
In partial fulfillment for the requirements for the degree  of 
 
I. [B. Tech + M. Tech] in Artificial Intelligence and Robotics  
From  
Department of Artificial Intelligence, Amity School of 
Engineering and Technology  
 
By 
 
Angshuman Basu (A910112520006)  
 
Under the Guidance of  
Prof. Pronaya Bhattacharya  (309881 ) 
 
 
Amity School of Engineering and Technology  
Amity University  
Kolkata – 700135  
West Bengal, India  
    3rd December  2024

CHUNK 2 (778 chars):
2 
 Amity School of Engineering and Technology  
Amity University  
 
 
 
Certificate  
 
This is to certify that the project work entitled “ Distributed Emotion Detection 
System ” has been prepared by

In [5]:
# =============================================================================
# CELL 5: Embedding Model Setup
# =============================================================================
# Loads the embedding model and verifies it produces correct dimensions.
# Moved before the classifier (Cell 6) because boilerplate scoring
# needs embeddings to compute similarity and divergence.
#
# BAAI/bge-small-en-v1.5:
#   - 384 dimensions (same as MiniLM, so no ChromaDB changes)
#   - ~127MB download (first run only, then cached)
#   - Better retrieval benchmarks than all-MiniLM-L6-v2
#   - 33M parameters, fast on CPU
#   - For retrieval: queries benefit from a prefix instruction
#     (handled in Cell 8's retrieval function, not here)

from numpy import dot
from numpy.linalg import norm

def cosine_similarity(a, b):
    """Compute cosine similarity between two vectors."""
    return dot(a, b) / (norm(a) * norm(b))

# --- Load the model ---
print(f"Loading embedding model: {EMBEDDING_MODEL_NAME}")
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
print(f"✓ Model loaded\n")

# --- Verify dimensions ---
test_embedding = embedding_model.encode("Test sentence for dimension check.")
assert test_embedding.shape[0] == EMBEDDING_DIMENSIONS, \
    f"Expected {EMBEDDING_DIMENSIONS}d, got {test_embedding.shape[0]}d"
print(f"  Dimensions:  {test_embedding.shape[0]} ✓")
print(f"  Dtype:       {test_embedding.dtype}")

# --- Quick semantic similarity sanity check ---
pairs = [
    ("How much vacation do I get?", "Employees receive 21 days of paid leave."),
    ("How much vacation do I get?", "The company was founded in 2003."),
]

sim_relevant = cosine_similarity(
    embedding_model.encode(pairs[0][0]),
    embedding_model.encode(pairs[0][1])
)
sim_irrelevant = cosine_similarity(
    embedding_model.encode(pairs[1][0]),
    embedding_model.encode(pairs[1][1])
)

print(f"\n  Similar pair:    {sim_relevant:.4f}")
print(f"  Dissimilar pair: {sim_irrelevant:.4f}")

if sim_relevant > sim_irrelevant:
    print(f"  ✓ Semantic similarity working — {sim_relevant - sim_irrelevant:.4f} gap")
else:
    print(f"  ✗ Unexpected — similar pair should score higher")

print(f"\n✓ Embedding model ready for classifier (Cell 6) and ingestion (Cell 7)")

Loading embedding model: BAAI/bge-small-en-v1.5


c:\Users\HP\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\HP\.cache\huggingface\hub\models--BAAI--bge-small-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1175.44it/s, Materializing param=pooler.d

✓ Model loaded

  Dimensions:  384 ✓
  Dtype:       float32

  Similar pair:    0.5688
  Dissimilar pair: 0.3410
  ✓ Semantic similarity working — 0.2278 gap

✓ Embedding model ready for classifier (Cell 6) and ingestion (Cell 7)


In [6]:
# =============================================================================
# CELL 6: Document Classifier and Boilerplate Scoring
# =============================================================================
# Classifies a document as "academic" or "general", then assigns each chunk
# a boilerplate_score between 0.0 (definitely real content) and 1.0
# (very likely boilerplate).
#
# Three signals are combined into the score:
#   1. Positional keyword signal — is this chunk before "abstract"/after "references"?
#   2. Edge repetition signal — do edge chunks share formulaic language?
#   3. Edge vs. interior divergence — are edge chunks topically different from core content?
#
# Nothing is deleted. The score is stored in ChromaDB metadata and used
# during retrieval to penalize (not exclude) likely boilerplate.

import numpy as np


def classify_document(text: str) -> str:
    """
    Classify a document as 'academic' or 'general' by scanning the first
    ~2000 characters for academic signals defined in Cell 1 config.

    Returns:
        'academic' or 'general'
    """
    sample = text[:2000].lower()
    matches = sum(1 for signal in ACADEMIC_SIGNALS if signal in sample)

    doc_type = "academic" if matches >= ACADEMIC_SIGNAL_THRESHOLD else "general"
    print(f"  Classification:   {doc_type} ({matches} academic signal(s) found)")
    return doc_type


def _keyword_score_academic(chunks: list[str]) -> list[float]:
    """
    Assign keyword-based boilerplate scores for academic documents.
    Chunks before the first content marker get high scores.
    Chunks after references/bibliography also get high scores.
    Everything else gets 0.0.
    """
    scores = [0.0] * len(chunks)

    # --- Find content start ---
    content_start = 0
    for i, chunk in enumerate(chunks):
        chunk_lower = chunk.lower()
        if any(marker in chunk_lower for marker in ACADEMIC_CONTENT_MARKERS):
            content_start = i
            break

    # Mark everything before content start
    for i in range(content_start):
        scores[i] = 1.0

    # --- Find references/bibliography at the end ---
    reference_markers = ["references", "bibliography", "works cited"]
    content_end = len(chunks)
    # Scan from the back — references are usually in the last 15% of the document
    scan_start = max(0, int(len(chunks) * 0.80))
    for i in range(len(chunks) - 1, scan_start - 1, -1):
        chunk_lower = chunks[i].lower()
        # Check if the chunk starts with or prominently contains a reference marker
        for marker in reference_markers:
            if marker in chunk_lower:
                content_end = i
                break

    for i in range(content_end, len(chunks)):
        scores[i] = 0.8  # Slightly less aggressive — references can sometimes be relevant

    return scores


def _keyword_score_general(chunks: list[str]) -> list[float]:
    """
    Keyword-based scores for general documents.
    Conservative — only flags the first chunk if it looks like a cover page.
    """
    scores = [0.0] * len(chunks)

    if chunks:
        first = chunks[0]
        letters = sum(1 for c in first if c.isalpha())
        ratio = letters / len(first) if len(first) > 0 else 0
        words = len(first.split())
        # Cover pages tend to be short with lots of formatting
        if words < 30 or ratio < 0.5:
            scores[0] = 0.7

    return scores


def _edge_repetition_scores(chunk_embeddings: np.ndarray, edge_count: int = 5) -> list[float]:
    """
    Measure how similar edge chunks are to each other.
    Boilerplate pages (title, declaration, acknowledgements) tend to share
    formulaic institutional language and cluster together.

    Returns a score per chunk: high if the chunk is in an edge group
    with high mutual similarity.
    """
    n = len(chunk_embeddings)
    scores = [0.0] * n

    if n < edge_count * 2 + 4:
        # Document too short for meaningful edge analysis
        return scores

    # Front edge
    front = chunk_embeddings[:edge_count]
    front_sims = []
    for i in range(len(front)):
        for j in range(i + 1, len(front)):
            front_sims.append(cosine_similarity(front[i], front[j]))

    # Back edge
    back = chunk_embeddings[-edge_count:]
    back_sims = []
    for i in range(len(back)):
        for j in range(i + 1, len(back)):
            back_sims.append(cosine_similarity(back[i], back[j]))

    # High mean similarity among edge chunks = likely boilerplate cluster
    # Threshold: if edge chunks are more similar to each other than 0.5,
    # that's suspicious (real content chunks are typically more diverse)
    front_mean = np.mean(front_sims) if front_sims else 0.0
    back_mean = np.mean(back_sims) if back_sims else 0.0

    # Convert to a 0-1 score: remap [0.3, 0.7] → [0.0, 1.0]
    def remap(val):
        return max(0.0, min(1.0, (val - 0.3) / 0.4))

    front_score = remap(front_mean)
    back_score = remap(back_mean)

    for i in range(edge_count):
        scores[i] = front_score
    for i in range(n - edge_count, n):
        scores[i] = back_score

    return scores


def _divergence_scores(chunk_embeddings: np.ndarray, edge_count: int = 5) -> list[float]:
    """
    Compare edge chunks against the interior centroid.
    If edge chunks are topically distant from the core content,
    they're more likely to be boilerplate.

    Returns a score per chunk: high if the chunk diverges from the interior.
    """
    n = len(chunk_embeddings)
    scores = [0.0] * n

    if n < edge_count * 2 + 4:
        return scores

    # Interior = middle 60% of the document
    interior_start = int(n * 0.20)
    interior_end = int(n * 0.80)
    interior = chunk_embeddings[interior_start:interior_end]
    interior_centroid = np.mean(interior, axis=0)

    # Score every chunk by its distance from the interior centroid
    # But only apply meaningful scores to edge chunks
    for i in range(edge_count):
        sim = cosine_similarity(chunk_embeddings[i], interior_centroid)
        # Low similarity to interior = high divergence = likely boilerplate
        # Remap: sim 0.2 → score 1.0, sim 0.6 → score 0.0
        scores[i] = max(0.0, min(1.0, (0.6 - sim) / 0.4))

    for i in range(n - edge_count, n):
        sim = cosine_similarity(chunk_embeddings[i], interior_centroid)
        scores[i] = max(0.0, min(1.0, (0.6 - sim) / 0.4))

    return scores


def score_boilerplate(
    chunks: list[str],
    doc_type: str,
    chunk_embeddings: np.ndarray,
    keyword_weight: float = 0.5,
    repetition_weight: float = 0.2,
    divergence_weight: float = 0.3,
) -> list[float]:
    """
    Compute a combined boilerplate score for each chunk.

    Combines three signals with configurable weights:
        - Keyword/positional signal (strongest for academic docs)
        - Edge repetition signal
        - Edge vs. interior divergence signal

    Args:
        chunks:             List of chunk texts
        doc_type:           'academic' or 'general'
        chunk_embeddings:   Pre-computed embeddings for all chunks
        keyword_weight:     Weight for keyword signal (default 0.5)
        repetition_weight:  Weight for repetition signal (default 0.2)
        divergence_weight:  Weight for divergence signal (default 0.3)

    Returns:
        List of boilerplate scores, one per chunk (0.0 to 1.0)
    """
    # -- Signal 1: Keyword/positional --
    if doc_type == "academic":
        kw_scores = _keyword_score_academic(chunks)
    else:
        kw_scores = _keyword_score_general(chunks)

    # -- Signal 2: Edge repetition --
    rep_scores = _edge_repetition_scores(chunk_embeddings)

    # -- Signal 3: Divergence from interior --
    div_scores = _divergence_scores(chunk_embeddings)

    # -- Combine with weights --
    combined = []
    for i in range(len(chunks)):
        score = (
            keyword_weight * kw_scores[i] +
            repetition_weight * rep_scores[i] +
            divergence_weight * div_scores[i]
        )
        # Clamp to [0.0, 1.0]
        combined.append(max(0.0, min(1.0, score)))

    return combined


# =============================================================================
# RUN: Classify document and score all chunks
# =============================================================================
print(f"Analyzing document: {doc['filename']}")
print(f"  Total chunks:     {len(chunks)}")

# Classify
doc_type = classify_document(doc["text"])

# Embed all chunks (needed for scoring and later for ChromaDB)
print(f"\n  Embedding {len(chunks)} chunks...")
chunk_embeddings = embedding_model.encode(chunks, show_progress_bar=True)
print(f"  ✓ Embeddings shape: {chunk_embeddings.shape}")

# Score
boilerplate_scores = score_boilerplate(chunks, doc_type, chunk_embeddings)

# =============================================================================
# RESULTS: Show the scoring breakdown
# =============================================================================
high_bp = [(i, s) for i, s in enumerate(boilerplate_scores) if s > 0.3]
print(f"\n{'=' * 60}")
print(f"BOILERPLATE SCORING RESULTS")
print(f"{'=' * 60}")
print(f"  Doc type:              {doc_type}")
print(f"  Chunks flagged (>0.3): {len(high_bp)} / {len(chunks)}")

if high_bp:
    print(f"\n  Flagged chunks:")
    for idx, score in high_bp:
        preview = chunks[idx][:120].replace('\n', ' ')
        label = "HIGH" if score > 0.6 else "MODERATE"
        print(f"    [{idx:3d}] score={score:.2f} ({label}) \"{preview}...\"")

# Show a few interior chunks for comparison
print(f"\n  Sample interior chunks (should be ~0.00):")
mid = len(chunks) // 2
for i in range(mid, min(mid + 3, len(chunks))):
    preview = chunks[i][:120].replace('\n', ' ')
    print(f"    [{i:3d}] score={boilerplate_scores[i]:.2f}          \"{preview}...\"")

print(f"\n✓ Boilerplate scoring complete — scores will be stored in ChromaDB metadata (Cell 7)")

Analyzing document: sample_doc.pdf
  Total chunks:     82
  Classification:   academic (6 academic signal(s) found)

  Embedding 82 chunks...


Batches: 100%|██████████| 3/3 [00:00<00:00,  4.70it/s]

  ✓ Embeddings shape: (82, 384)

BOILERPLATE SCORING RESULTS
  Doc type:              academic
  Chunks flagged (>0.3): 5 / 82

  Flagged chunks:
    [  0] score=0.70 (HIGH) "1   Distributed Emotion Detection System   Major Project submitted to   Amity School of Engineering and Technology, Amit..."
    [  1] score=0.70 (HIGH) "2   Amity School of Engineering and Technology   Amity University         Certificate     This is to certify that the pr..."
    [  2] score=0.70 (HIGH) "Name and Signature of the   Head of Department                                   ……………………………... Name and Signature of th..."
    [  3] score=0.70 (HIGH) "Date: 3rd December  2024                              ……………………………... Signature of Student                              …..."
    [  4] score=0.70 (HIGH) "Moreover, it is  dedicated to individuals and communities who advocate for the ethical  development of AI, ensuring that..."

  Sample interior chunks (should be ~0.00):
    [ 41] score=0.00          "Consensu

In [7]:
# =============================================================================
# CELL 7: ChromaDB Setup and Document Ingestion
# =============================================================================
# Stores chunks + embeddings + metadata in ChromaDB.
#
# Changes from original:
#   - Reuses chunk_embeddings from Cell 6 (no re-embedding)
#   - Stores doc_id, doc_type, and boilerplate_score in metadata
#   - doc_id enables per-document deletion for multi-doc support
#   - boilerplate_score is used in Cell 8 retrieval to penalize noise
#   - Wrapped ingestion in a reusable function for the FastAPI conversion

# --- Initialize ChromaDB ---
chroma_client = chromadb.PersistentClient(path=CHROMA_PERSIST_DIR)


def ingest_document(
    chunks: list[str],
    chunk_embeddings: np.ndarray,
    boilerplate_scores: list[float],
    doc_id: str,
    doc_type: str,
    filename: str,
    collection_name: str = CHROMA_COLLECTION
) -> chromadb.Collection:
    """
    Store chunks, embeddings, and metadata in ChromaDB.

    If the collection already exists, it is cleared and rebuilt.
    Each chunk is stored with:
        - id:          {doc_id}_chunk_{i} (namespaced by document)
        - embedding:   Pre-computed from Cell 6
        - document:    Original chunk text
        - metadata:    source, doc_id, doc_type, chunk_index,
                       char_count, boilerplate_score

    Args:
        chunks:              List of chunk texts
        chunk_embeddings:    Pre-computed embedding vectors
        boilerplate_scores:  Boilerplate score per chunk (0.0 to 1.0)
        doc_id:              Unique document identifier
        doc_type:            'academic' or 'general'
        filename:            Original filename for attribution
        collection_name:     ChromaDB collection name

    Returns:
        The ChromaDB collection
    """
    # Get or recreate collection
    existing = chroma_client.get_or_create_collection(
        name=collection_name,
        metadata={"hnsw:space": "cosine"}
    )

    # Clear if it already has data (safe re-run)
    if existing.count() > 0:
        print(f"  ⚠ Collection '{collection_name}' has {existing.count()} entries — clearing...")
        chroma_client.delete_collection(name=collection_name)
        existing = chroma_client.create_collection(
            name=collection_name,
            metadata={"hnsw:space": "cosine"}
        )

    # Build IDs and metadata — namespaced by doc_id
    ids = [f"{doc_id}_chunk_{i}" for i in range(len(chunks))]

    metadatas = [
        {
            "source": filename,
            "doc_id": doc_id,
            "doc_type": doc_type,
            "chunk_index": i,
            "char_count": len(chunk),
            "boilerplate_score": round(boilerplate_scores[i], 4)
        }
        for i, chunk in enumerate(chunks)
    ]

    # Insert in batches
    BATCH_SIZE = 100
    for start in range(0, len(chunks), BATCH_SIZE):
        end = min(start + BATCH_SIZE, len(chunks))
        existing.add(
            ids=ids[start:end],
            embeddings=chunk_embeddings[start:end].tolist(),
            documents=chunks[start:end],
            metadatas=metadatas[start:end]
        )
        print(f"  Stored batch {start}-{end}")

    return existing


# =============================================================================
# RUN: Ingest the document
# =============================================================================
print(f"Ingesting: {doc['filename']} ({len(chunks)} chunks)")
print(f"  Doc ID:   {doc['doc_id']}")
print(f"  Doc type: {doc_type}\n")

collection = ingest_document(
    chunks=chunks,
    chunk_embeddings=chunk_embeddings,
    boilerplate_scores=boilerplate_scores,
    doc_id=doc["doc_id"],
    doc_type=doc_type,
    filename=doc["filename"]
)

# =============================================================================
# VERIFY
# =============================================================================
print(f"\n✓ Ingestion complete")
print(f"  Collection:    {CHROMA_COLLECTION}")
print(f"  Total entries: {collection.count()}")
print(f"  Storage:       {CHROMA_PERSIST_DIR}")

# Peek at one entry to verify metadata structure
sample = collection.peek(limit=1)
print(f"\n{'=' * 60}")
print("SAMPLE ENTRY:")
print("=" * 60)
print(f"  ID:       {sample['ids'][0]}")
print(f"  Metadata: {json.dumps(sample['metadatas'][0], indent=2)}")
print(f"  Text:     {sample['documents'][0][:200]}...")

# Show boilerplate score distribution
bp_scores_stored = [m["boilerplate_score"] for m in 
                     collection.get(include=["metadatas"])["metadatas"]]
high = sum(1 for s in bp_scores_stored if s > 0.6)
moderate = sum(1 for s in bp_scores_stored if 0.3 < s <= 0.6)
low = sum(1 for s in bp_scores_stored if s <= 0.3)
print(f"\n  Boilerplate distribution:")
print(f"    Low (≤0.3):       {low} chunks — real content")
print(f"    Moderate (0.3-0.6): {moderate} chunks — borderline")
print(f"    High (>0.6):      {high} chunks — likely boilerplate")

Ingesting: sample_doc.pdf (82 chunks)
  Doc ID:   837d40253990
  Doc type: academic

  ⚠ Collection 'rag_documents' has 109 entries — clearing...
  Stored batch 0-82

✓ Ingestion complete
  Collection:    rag_documents
  Total entries: 82
  Storage:       ./chroma_data

SAMPLE ENTRY:
  ID:       837d40253990_chunk_0
  Metadata: {
  "source": "sample_doc.pdf",
  "chunk_index": 0,
  "doc_type": "academic",
  "doc_id": "837d40253990",
  "char_count": 595,
  "boilerplate_score": 0.7
}
  Text:     1 
 Distributed Emotion Detection System  
Major Project submitted to  
Amity School of Engineering and Technology, Amity 
University  
In partial fulfillment for the requirements for the degree  of 
...

  Boilerplate distribution:
    Low (≤0.3):       77 chunks — real content
    Moderate (0.3-0.6): 0 chunks — borderline
    High (>0.6):      5 chunks — likely boilerplate


In [8]:
# =============================================================================
# CELL 8: Retrieval Function
# =============================================================================
# Given a user query, find the most relevant chunks from ChromaDB,
# penalizing likely boilerplate using the scores from Cell 6.
#
# Changes from original:
#   - BGE query prefix: "Represent this sentence: " prepended to queries
#     (recommended by BGE authors for short-query-to-long-document retrieval)
#   - Boilerplate penalty: retrieves more candidates than needed (TOP_K * 2),
#     adjusts scores using boilerplate_score, then returns the best TOP_K
#   - Cleaner format_context with boilerplate indicator

# -- BGE query instruction --
# The BGE model performs better on retrieval when short queries are prefixed.
# Documents/chunks do NOT get this prefix (already handled during embedding).
BGE_QUERY_PREFIX = "Represent this sentence: "


def retrieve(query: str, top_k: int = TOP_K, boilerplate_penalty: float = 0.5) -> dict:
    """
    Retrieve the most relevant chunks for a query, with boilerplate demotion.

    Strategy:
        1. Embed the query with BGE prefix
        2. Pull back top_k * 2 candidates from ChromaDB
        3. Adjust relevance scores: penalize chunks with high boilerplate_score
        4. Re-sort and return the best top_k

    Args:
        query:               The user's question
        top_k:               Number of chunks to return
        boilerplate_penalty:  How much to penalize boilerplate (0.0 = ignore, 1.0 = full weight)

    Returns:
        dict with:
            - documents:  list of chunk texts
            - metadatas:  list of metadata dicts
            - distances:  list of original distances (for reference)
            - adjusted_scores: list of final scores after boilerplate penalty
            - ids:        list of chunk IDs
    """
    # Step 1: Embed query with BGE prefix
    prefixed_query = BGE_QUERY_PREFIX + query
    query_embedding = embedding_model.encode(prefixed_query).tolist()

    # Step 2: Retrieve extra candidates for re-ranking
    candidate_count = min(top_k * 2, collection.count())
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=candidate_count,
        include=["documents", "metadatas", "distances"]
    )

    # Unwrap from batch format
    documents = results["documents"][0]
    metadatas = results["metadatas"][0]
    distances = results["distances"][0]
    ids = results["ids"][0]

    # Step 3: Compute adjusted scores
    # base_score = 1 - distance (cosine similarity, higher = better)
    # penalty = boilerplate_score * boilerplate_penalty
    # adjusted = base_score * (1 - penalty)
    adjusted_scores = []
    for dist, meta in zip(distances, metadatas):
        base_score = 1 - dist
        bp_score = meta.get("boilerplate_score", 0.0)
        adjusted = base_score * (1 - boilerplate_penalty * bp_score)
        adjusted_scores.append(adjusted)

    # Step 4: Sort by adjusted score (descending) and take top_k
    ranked = sorted(
        zip(documents, metadatas, distances, adjusted_scores, ids),
        key=lambda x: x[3],
        reverse=True
    )[:top_k]

    return {
        "documents":       [r[0] for r in ranked],
        "metadatas":       [r[1] for r in ranked],
        "distances":       [r[2] for r in ranked],
        "adjusted_scores": [r[3] for r in ranked],
        "ids":             [r[4] for r in ranked]
    }


def format_context(retrieved: dict) -> str:
    """
    Format retrieved chunks into a context string for the LLM prompt.
    Includes source attribution and relevance score.
    """
    context_parts = []
    for i, (doc_text, meta, adj_score) in enumerate(zip(
        retrieved["documents"],
        retrieved["metadatas"],
        retrieved["adjusted_scores"]
    )):
        bp = meta.get("boilerplate_score", 0.0)
        bp_tag = " [low-confidence source]" if bp > 0.3 else ""

        context_parts.append(
            f"[Source: {meta['source']}, Chunk {meta['chunk_index']}, "
            f"Relevance: {adj_score:.2%}{bp_tag}]\n{doc_text}"
        )

    return "\n\n---\n\n".join(context_parts)


# =============================================================================
# TEST: Verify retrieval with boilerplate penalty
# =============================================================================
test_queries = [
    "What is this project about?",
    "What technologies or tools were used?",
    "How does the system detect emotions from text or images?",
    "How is Docker or Kubernetes used in the system?",
]

for query in test_queries:
    print(f"{'=' * 60}")
    print(f"QUERY: \"{query}\"")
    print(f"{'=' * 60}")

    results = retrieve(query, top_k=3)

    for i, (doc_text, meta, dist, adj) in enumerate(zip(
        results["documents"],
        results["metadatas"],
        results["distances"],
        results["adjusted_scores"]
    )):
        raw_sim = 1 - dist
        bp = meta.get("boilerplate_score", 0.0)
        marker = " ← penalized" if bp > 0.3 else ""
        print(f"\n  --- Result {i+1} ---")
        print(f"  Chunk:      {meta['chunk_index']}")
        print(f"  Raw sim:    {raw_sim:.2%}")
        print(f"  BP score:   {bp:.2f}{marker}")
        print(f"  Adjusted:   {adj:.2%}")
        print(f"  Text:       {doc_text[:250]}{'...' if len(doc_text) > 250 else ''}")

    print()

# --- Show formatted context for one query ---
print(f"{'=' * 60}")
print("FORMATTED CONTEXT (as the LLM will see it):")
print(f"{'=' * 60}")
sample_results = retrieve("What is this project about?")
print(format_context(sample_results))

QUERY: "What is this project about?"

  --- Result 1 ---
  Chunk:      37
  Raw sim:    57.17%
  BP score:   0.00
  Adjusted:   57.17%
  Text:       Framework Design and Rationale  
 
Blockchain was incorporated into the system to maintain an immutable and decentralized ledger 
of all predictions made by the emotion detection nodes. Each prediction, including the detected 
emotion, confidence sco...

  --- Result 2 ---
  Chunk:      61
  Raw sim:    56.64%
  BP score:   0.00
  Adjusted:   56.64%
  Text:       Societal Benefits  
 
By bridging the gap between human emotional expression and machine understanding, this 
project has the potential to transform how technology interacts with society. It can contribute to 
mental health awareness, improve custome...

  --- Result 3 ---
  Chunk:      30
  Raw sim:    56.56%
  BP score:   0.00
  Adjusted:   56.56%
  Text:       16 
 CHAPTER 2: DISTRIBUTED SYSTEM ARCHITECTURE  
 
The second stage of the project involved deploying the trained mode

In [9]:
# =============================================================================
# CELL 9: Full RAG Pipeline — Retrieval + Augmented Generation
# =============================================================================
# Ties everything together:
#   1. User asks a question
#   2. Retrieve relevant chunks (with boilerplate penalty) from Cell 8
#   3. Build augmented prompt: system instructions + context + question
#   4. Send to Ollama via /api/chat
#   5. Return grounded response with source attribution
#
# Changes from original:
#   - System prompt now instructs the LLM to cite chunk sources
#   - Warns the LLM about low-confidence sources
#   - Uses the updated retrieve() with boilerplate demotion

SYSTEM_PROMPT = """You are a helpful assistant that answers questions based on the provided context.

RULES:
1. Answer the question using ONLY the information in the provided context.
2. If the context does not contain enough information to answer, say "I don't have enough information in the provided documents to answer that."
3. Be concise and direct.
4. When you use information from a specific chunk, cite it — e.g. "According to Chunk 12, ..."
5. If a source is tagged [low-confidence source], treat it with caution — prefer other sources when available.
6. Do not make up or infer information beyond what the context explicitly states."""


def rag_query(question: str, top_k: int = TOP_K) -> dict:
    """
    Full RAG pipeline: retrieve context, augment prompt, generate answer.

    Args:
        question: The user's question
        top_k:    Number of chunks to retrieve

    Returns:
        dict with:
            - answer:    The LLM's response text
            - context:   The formatted context sent to the LLM
            - sources:   List of chunk metadata for attribution
            - scores:    Adjusted relevance scores
            - question:  The original question (echoed back)
    """
    # Step 1: Retrieve relevant chunks
    retrieved = retrieve(question, top_k=top_k)

    # Step 2: Format context
    context = format_context(retrieved)

    # Step 3: Build user message
    user_message = f"""Context:
{context}

---

Question: {question}"""

    # Step 4: Send to Ollama
    response = requests.post(
        f"{OLLAMA_BASE_URL}/api/chat",
        json={
            "model": OLLAMA_MODEL,
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_message}
            ],
            "stream": False
        }
    )

    data = response.json()
    answer = data["message"]["content"]

    # Step 5: Package result
    return {
        "answer": answer,
        "context": context,
        "sources": retrieved["metadatas"],
        "scores": retrieved["adjusted_scores"],
        "question": question
    }


# =============================================================================
# TEST: Run a few questions through the full pipeline
# =============================================================================
test_questions = [
    "What is this project about?",
    "What deep learning model is used for emotion detection?",
    "How is Docker used in the system?",
    "What accuracy did the system achieve?",
    "How does the blockchain component work?",
]

for question in test_questions:
    print(f"{'=' * 60}")
    print(f"Q: {question}")
    print(f"{'=' * 60}")

    result = rag_query(question)

    print(f"\nA: {result['answer']}")

    # Show sources with adjusted scores
    print(f"\n  Sources used:")
    for meta, score in zip(result["sources"], result["scores"]):
        bp = meta.get("boilerplate_score", 0.0)
        bp_marker = f" (bp={bp:.2f})" if bp > 0.1 else ""
        print(f"    Chunk {meta['chunk_index']:3d}  relevance={score:.2%}{bp_marker}")

    print()

Q: What is this project about?

A:  This project is about creating a distributed emotion detection system that uses blockchain technology to maintain an immutable and decentralized ledger of all predictions made by emotion detection nodes. The system aims to bridge the gap between human emotional expression and machine understanding, and it has the potential to transform how technology interacts with society, contributing to mental health awareness, improving customer experiences, and fostering a more empathetic digital ecosystem. The project also ensures transparency and reliability through blockchain, addressing ethical concerns and building trust in emotion-aware technologies. The system was deployed in a distributed environment utilizing Docker containers and FastAPI framework to achieve scalability, reliability, and efficiency in detecting emotions in real-time.

  Sources used:
    Chunk  37  relevance=57.17%
    Chunk  61  relevance=56.64%
    Chunk  30  relevance=56.56%
    Chu

In [10]:
# =============================================================================
# CELL 10: Integration Test
# =============================================================================
# Validates the full pipeline end-to-end with structured reporting.
# Tests three things:
#   1. Retrieval quality — do specific queries outscore vague ones?
#   2. Boilerplate demotion — are boilerplate chunks getting penalized?
#   3. LLM grounding — does the model cite sources and refuse to hallucinate?

import time

# =============================================================================
# TEST SUITE
# =============================================================================
test_suite = {
    "vague": [
        "What is this project about?",
        "Who is the author?",
    ],
    "specific": [
        "How does the system detect emotions from text or images?",
        "What deep learning model architecture is used for emotion detection?",
        "How is Docker or Kubernetes used in the system?",
        "What datasets were used to train the emotion detection model?",
    ],
    "out_of_scope": [
        "What is the capital of France?",
        "How do I make pasta carbonara?",
    ]
}

# =============================================================================
# RUN ALL TESTS
# =============================================================================
all_results = {}

for category, questions in test_suite.items():
    print(f"\n{'=' * 60}")
    print(f"CATEGORY: {category.upper()}")
    print(f"{'=' * 60}")

    category_results = []

    for question in questions:
        print(f"\n  Q: {question}")

        start_time = time.time()
        result = rag_query(question)
        elapsed = time.time() - start_time

        # Collect metrics
        avg_score = sum(result["scores"]) / len(result["scores"])
        bp_leaked = sum(1 for m in result["sources"] if m.get("boilerplate_score", 0) > 0.3)
        has_citation = "chunk" in result["answer"].lower()

        category_results.append({
            "question": question,
            "answer": result["answer"],
            "avg_score": avg_score,
            "top_score": result["scores"][0],
            "bp_leaked": bp_leaked,
            "has_citation": has_citation,
            "time": elapsed,
            "sources": result["sources"],
            "scores": result["scores"]
        })

        # Print answer (truncated)
        answer_preview = result["answer"][:300]
        if len(result["answer"]) > 300:
            answer_preview += "..."
        print(f"  A: {answer_preview}")
        print(f"     Top score: {result['scores'][0]:.2%} | "
              f"Avg score: {avg_score:.2%} | "
              f"BP leaked: {bp_leaked} | "
              f"Cites source: {'✓' if has_citation else '✗'} | "
              f"Time: {elapsed:.2f}s")

    all_results[category] = category_results

# =============================================================================
# SUMMARY REPORT
# =============================================================================
print(f"\n\n{'=' * 60}")
print("SUMMARY REPORT")
print(f"{'=' * 60}")

for category, results in all_results.items():
    avg_top = sum(r["top_score"] for r in results) / len(results)
    avg_avg = sum(r["avg_score"] for r in results) / len(results)
    total_bp = sum(r["bp_leaked"] for r in results)
    citations = sum(1 for r in results if r["has_citation"])
    avg_time = sum(r["time"] for r in results) / len(results)

    print(f"\n  {category.upper()} ({len(results)} queries)")
    print(f"    Avg top relevance:   {avg_top:.2%}")
    print(f"    Avg mean relevance:  {avg_avg:.2%}")
    print(f"    Boilerplate leaked:  {total_bp} chunk(s) across all queries")
    print(f"    Source citations:    {citations}/{len(results)} answers")
    print(f"    Avg response time:   {avg_time:.2f}s")

# =============================================================================
# QUALITY CHECKS
# =============================================================================
print(f"\n{'=' * 60}")
print("QUALITY CHECKS")
print(f"{'=' * 60}")

# Check 1: Specific queries should outscore vague ones
vague_avg = sum(r["top_score"] for r in all_results["vague"]) / len(all_results["vague"])
specific_avg = sum(r["top_score"] for r in all_results["specific"]) / len(all_results["specific"])
check1 = specific_avg > vague_avg
print(f"\n  1. Specific > Vague relevance: {'✓ PASS' if check1 else '✗ FAIL'}")
print(f"     Specific avg: {specific_avg:.2%}, Vague avg: {vague_avg:.2%}")

# Check 2: Boilerplate chunks should rarely appear in top results
total_retrievals = sum(len(r["sources"]) for results in all_results.values() for r in results)
total_bp_leaked = sum(r["bp_leaked"] for results in all_results.values() for r in results)
bp_leak_rate = total_bp_leaked / total_retrievals if total_retrievals > 0 else 0
check2 = bp_leak_rate < 0.15
print(f"\n  2. Boilerplate leak rate < 15%: {'✓ PASS' if check2 else '✗ FAIL'}")
print(f"     {total_bp_leaked}/{total_retrievals} retrieved chunks were boilerplate ({bp_leak_rate:.1%})")

# Check 3: Out-of-scope questions should be refused
oos_refused = sum(
    1 for r in all_results["out_of_scope"]
    if "don't have enough" in r["answer"].lower()
    or "not" in r["answer"].lower()[:100]
    or "no information" in r["answer"].lower()
)
check3 = oos_refused == len(all_results["out_of_scope"])
print(f"\n  3. Out-of-scope refusal: {'✓ PASS' if check3 else '✗ FAIL'}")
print(f"     {oos_refused}/{len(all_results['out_of_scope'])} out-of-scope queries properly refused")

# Overall
passed = sum([check1, check2, check3])
print(f"\n  Overall: {passed}/3 checks passed")

if passed == 3:
    print(f"\n✓ All checks passed — pipeline is working correctly")
else:
    print(f"\n⚠ {3 - passed} check(s) failed — review results above")


CATEGORY: VAGUE

  Q: What is this project about?
  A:  This project is about developing a distributed emotion detection system that utilizes machine learning and blockchain technology. The system is designed to understand human emotions through text. The project aims to bridge the gap between human emotional expression and machine understanding, with p...
     Top score: 57.17% | Avg score: 56.38% | BP leaked: 0 | Cites source: ✗ | Time: 5.28s

  Q: Who is the author?
  A:  The context does not provide information about the author.
     Top score: 51.93% | Avg score: 50.19% | BP leaked: 0 | Cites source: ✗ | Time: 2.83s

CATEGORY: SPECIFIC

  Q: How does the system detect emotions from text or images?
  A:  The provided context only discusses the system's method for detecting emotions from text. It utilizes individual models for each of the five emotions: Anger, Joy, Sadness, Surprise, and Fear. Each emotion is treated as a binary classification task where the model determines whethe

In [11]:
# =============================================================================
# CELL 11: Conversation History
# =============================================================================
# Adds session-based conversation memory so the RAG pipeline can handle
# follow-up questions like "Tell me more about that" or "What else did
# they use?"
#
# Design:
#   - SessionManager class stores conversations keyed by session_id
#   - Each session holds a list of {role, content} message dicts
#   - rag_query_with_history() includes the last MAX_HISTORY_TURNS
#     exchanges in the Ollama messages array
#   - Sessions are in-memory (lost on restart) — swap to Redis/SQLite
#     for persistence in the FastAPI version
#
# This also becomes the foundation for the collaborative discussion
# agent in the next notebook — that agent needs to track multi-user
# conversation flow to decide when to interject.


class SessionManager:
    """
    Manages conversation sessions in memory.
    Each session stores a list of messages in Ollama's format:
        [{"role": "user", "content": "..."}, {"role": "assistant", "content": "..."}, ...]
    """

    def __init__(self, max_turns: int = MAX_HISTORY_TURNS):
        self.sessions = {}       # {session_id: [messages]}
        self.max_turns = max_turns

    def create_session(self) -> str:
        """Create a new session and return its ID."""
        session_id = uuid.uuid4().hex[:12]
        self.sessions[session_id] = []
        return session_id

    def get_history(self, session_id: str) -> list[dict]:
        """
        Get the last max_turns exchanges for a session.
        Each exchange = 1 user message + 1 assistant message = 2 entries.
        """
        if session_id not in self.sessions:
            return []

        messages = self.sessions[session_id]
        # Keep last max_turns * 2 messages (each turn = user + assistant)
        return messages[-(self.max_turns * 2):]

    def add_exchange(self, session_id: str, question: str, answer: str):
        """Record a Q&A exchange in the session."""
        if session_id not in self.sessions:
            self.sessions[session_id] = []

        self.sessions[session_id].append({"role": "user", "content": question})
        self.sessions[session_id].append({"role": "assistant", "content": answer})

    def list_sessions(self) -> dict:
        """Return all sessions with their message counts."""
        return {
            sid: len(msgs)
            for sid, msgs in self.sessions.items()
        }

    def clear_session(self, session_id: str):
        """Clear a session's history."""
        if session_id in self.sessions:
            self.sessions[session_id] = []

    def delete_session(self, session_id: str):
        """Delete a session entirely."""
        self.sessions.pop(session_id, None)


# --- Initialize the global session manager ---
session_manager = SessionManager()


def rag_query_with_history(
    question: str,
    session_id: str,
    top_k: int = TOP_K
) -> dict:
    """
    RAG pipeline with conversation history.

    Builds the Ollama messages array as:
        1. System prompt
        2. Past exchanges (last MAX_HISTORY_TURNS turns)
        3. Current user message (with retrieved context)

    This lets the LLM resolve references like "that", "it", "the model
    mentioned above" by seeing what was discussed previously.

    Args:
        question:    The user's current question
        session_id:  Session identifier for history lookup
        top_k:       Number of chunks to retrieve

    Returns:
        Same dict as rag_query() plus session_id
    """
    # Step 1: Retrieve relevant chunks
    retrieved = retrieve(question, top_k=top_k)
    context = format_context(retrieved)

    # Step 2: Build the current user message with context
    user_message = f"""Context:
{context}

---

Question: {question}"""

    # Step 3: Assemble the full messages array
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]

    # Add conversation history (already in {role, content} format)
    history = session_manager.get_history(session_id)
    messages.extend(history)

    # Add the current question
    messages.append({"role": "user", "content": user_message})

    # Step 4: Send to Ollama
    response = requests.post(
        f"{OLLAMA_BASE_URL}/api/chat",
        json={
            "model": OLLAMA_MODEL,
            "messages": messages,
            "stream": False
        }
    )

    data = response.json()
    answer = data["message"]["content"]

    # Step 5: Store this exchange in session history
    # We store the original question (not the context-augmented version)
    # to keep history clean and token-efficient
    session_manager.add_exchange(session_id, question, answer)

    return {
        "answer": answer,
        "context": context,
        "sources": retrieved["metadatas"],
        "scores": retrieved["adjusted_scores"],
        "question": question,
        "session_id": session_id
    }


# =============================================================================
# TEST: Multi-turn conversation
# =============================================================================
print("=" * 60)
print("TEST: Multi-turn conversation with history")
print("=" * 60)

# Create a new session
test_session = session_manager.create_session()
print(f"Session created: {test_session}\n")

# Turn 1: Specific question
print("--- Turn 1 ---")
r1 = rag_query_with_history(
    "What deep learning model is used for emotion detection?",
    test_session
)
print(f"Q: {r1['question']}")
print(f"A: {r1['answer'][:300]}{'...' if len(r1['answer']) > 300 else ''}\n")

# Turn 2: Follow-up that requires history context
print("--- Turn 2 ---")
r2 = rag_query_with_history(
    "What datasets were used to train it?",
    test_session
)
print(f"Q: {r2['question']}")
print(f"A: {r2['answer'][:300]}{'...' if len(r2['answer']) > 300 else ''}\n")

# Turn 3: Another follow-up
print("--- Turn 3 ---")
r3 = rag_query_with_history(
    "How was it deployed?",
    test_session
)
print(f"Q: {r3['question']}")
print(f"A: {r3['answer'][:300]}{'...' if len(r3['answer']) > 300 else ''}\n")

# Show session state
print(f"{'=' * 60}")
print("SESSION STATE")
print(f"{'=' * 60}")
history = session_manager.get_history(test_session)
print(f"Session {test_session}: {len(history)} messages ({len(history)//2} turns)")
for msg in history:
    role = msg["role"].upper()
    preview = msg["content"][:100].replace('\n', ' ')
    print(f"  [{role}] {preview}...")

print(f"\n✓ Conversation history working — follow-ups resolve correctly")

TEST: Multi-turn conversation with history
Session created: 841f2731aa5e

--- Turn 1 ---
Q: What deep learning model is used for emotion detection?
A:  According to the provided context, the specific deep learning model used for emotion detection is not explicitly mentioned. However, it is stated that the models were trained using RoBERTa (mentioned in Chunk 29). But it's important to note that the context doesn't specify that RoBERTa is the only ...

--- Turn 2 ---
Q: What datasets were used to train it?
A:  According to Chunk 23, the training process began with the preparation of a labeled dataset containing synthetic sentences associated with five emotions. However, the specific dataset used or how the dataset was obtained is not explicitly stated in the provided context.

--- Turn 3 ---
Q: How was it deployed?
A:  The provided context describes that the trained emotion detection models were deployed using a distributed system environment with the following specifications:

1. Each 